In [62]:
from dataclasses import dataclass
from dotenv import load_dotenv
from tqdm import tqdm
import wandb

import transformers
from transformers import AutoModelForCausalLM, GPT2LMHeadModel, GPT2Tokenizer

import torch
from torch.optim import AdamW
from torch.nn import CrossEntropyLoss

load_dotenv()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === load teacher model ===
model_name = "gpt2"

teacher_model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # <- this should get rid of some warnings

teacher_model.eval()

# === prepare new model ===
student_model = transformers.models.GPT2LMHeadModel(teacher_model.config)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
# === TRANING ===
# === data generation ===
def synthetic_batch_generator(batch_size=1, seq_len=20, num_batches=1):
    for _ in range(num_batches):
        with torch.inference_mode():
            ids = teacher_model.generate(
                input_ids=None,
                attention_mask=None,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=True,
                max_new_tokens=seq_len - 1,
                num_return_sequences=batch_size,
            )

        # attention from 0th pos to first pad_token appearing after the 0th pos
        # the first appearing pad_token is included
        # it's a bit overly clever designed
        attention_mask = torch.zeros_like(ids, dtype=bool)
        attention_mask[:, 1:] = (
            ids[:, :-1] != tokenizer.pad_token_id
        )  # basically shifts ids one to the right
        attention_mask[:, :2] = 1

        yield ids.clone(), attention_mask


# === traing ===
@dataclass
class TrainCfg:
    project: str = "teacher-student"
    num_epoch: int = 1
    num_batches: int = 10
    batch_size: int = 7  # exprimental max on 4x Tesla V100
    max_seq_len: int = 1024
    lr :float = 5e-5


def train(cfg: TrainCfg):

    wandb.ini(
        project="student-teacher",
        config={
            "num_epoch": cfg.num_epoch,
            "num_batches": cfg.num_batches,
            "batch_size": cfg.batch_size,
            "max_seq_len": cfg.max_seq_len,
            "optimizer": "AdamW"
            "lr": cfg.lr
        }
    )

    batch_size = cfg.batch_size
    seq_len = cfg.max_seq_len
    num_batches = cfg.num_batches

    # Move models to GPU if available

    teacher_model.to(device)
    student_model.to(device)

    # Optimizer
    optimizer = AdamW(student_model.parameters(), lr=cfg.lr)

    # Scaler
    # scaler = torch.cuda.amp.GradScaler()
    scaler = torch.amp.GradScaler(device.type)

    for epoch in range(cfg.num_epoch):
        print(f"=== epoch {epoch+1}/{cfg.num_epoch} ===")
        student_model.train()

        loss = 0
        for ids, attention_mask in tqdm(
            synthetic_batch_generator(
                batch_size=batch_size, seq_len=seq_len, num_batches=num_batches
            ),
            total=num_batches,
        ):
            labels = ids.clone()
            labels[attention_mask == 0] = -100  # ignore loss for paddedd tokens

            # Forward pass: student model
            with torch.amp.autocast(device.type):
                outputs = student_model(
                    ids, attention_mask=attention_mask, labels=labels
                )
                loss = outputs.loss

            # Backward pass
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        student_model.eval()
        print(f"Epoch {epoch}, last loss: {loss.item()}")
        print(
            tokenizer.decode(
                student_model.generate(
                    input_ids=None,
                    attention_mask=None,
                    pad_token_id=tokenizer.eos_token_id,
                )
            )
        )
        print(
            tokenizer.decode(
                student_model.generate(
                    input_ids=None,
                    attention_mask=None,
                    pad_token_id=tokenizer.eos_token_id,
                    do_sample=True,
                )
            )
        )


# === do train ===
train(
    TrainCfg(
        num_epoch=20,
        num_batches=5,
        batch_size=7,
        max_seq_len=1024,
    )
)

=== epoch 1/20 ===


  0%|          | 0/5 [00:00<?, ?it/s]The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
100%|██████████| 5/5 [00:35<00:00,  7.14s/it]
/venv/main/lib/python3.12/site-packages/transformers/generation/utils.py:1554: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Epoch 0, last loss: 9.894011497497559
['<|endoftext|>.\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n']
['<|endoftext|>\n.\n.,.520 the\nThousands a and\n the.\n.,.\n']
=== epoch 2/20 ===


100%|██████████| 5/5 [00:35<00:00,  7.06s/it]


Epoch 1, last loss: 9.379371643066406
['<|endoftext|>..\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n']
['<|endoftext|>.\n\n. the\n,\n\n\n. the. the.\n\n arising Tina\n']
=== epoch 3/20 ===


...

Epoch 17, last loss: 6.986416339874268
['<|endoftext|>\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n']
['<|endoftext|> or in the in the the of so the a for with not in,, will. I be']
=== epoch 19/20 ===


100%|██████████| 5/5 [00:35<00:00,  7.10s/it]


Epoch 18, last loss: 6.959497928619385
['<|endoftext|>\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n']
['<|endoftext|>, on ".,. or to. for.\n of to that on the of of the']
=== epoch 20/20 ===


100%|██████████| 5/5 [00:32<00:00,  6.58s/it]


Epoch 19, last loss: 6.827829360961914
['<|endoftext|>\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n']
['<|endoftext|> an the.\n\n\n has are an\n\n\n\n It\n or\n\n,\n']


In [42]:
device

device(type='cuda')

In [ ]:
x = tokenizer("Peter is tired. Sleep is what", return_tensors="pt", device=device)

ids = teacher_model.generate(
    input_ids=x.input_ids.to(device),
    attention_mask=x.attention_mask.to(device),
    pad_token_id=tokenizer.eos_token_id,
    max_new_tokens=5,
)
tokenizer.decode(ids[0])

"Peter is tired. Sleep is what he needs. He's"

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

token_id = tokenizer.encode(".")[0]


logits = (
    student_model(torch.tensor([[token_id, token_id]], device=device))
    .logits[0][-1]
    .to("cpu")
)

x, y = torch.topk(logits, 10)

print(tokenizer.decode(token_id))

for i in range(x.size(0)):
    print(round(x[i].item(), 2), tokenizer.decode(y[i]))

.
7.34 .
6.12 ,
5.96 

5.74  to
5.55  a
5.53  of
5.52  the
5.32  and
5.21  in
4.98  that


In [ ]:
teacher_model.generate("1+1")

In [2]:
import torch

print(torch.cuda.memory_summary())

|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   2457 MiB |   5812 MiB |   1214 GiB |   1212 GiB |
|       from large pool |   2454 MiB |   5808 MiB |   1111 GiB |   1109 GiB |
|       from small pool |      2 MiB |     28 MiB |    103 GiB |    103 GiB |
|---------------------------------------------------------------------------|
| Active memory         |   2457 MiB |   5812 MiB |   1214 GiB |   1212 GiB |
|       from large pool |   2454 MiB |   5808 MiB |   1111 GiB |